In [1]:
import pandas as pd

In [2]:
# data
df = pd.read_csv("data1.csv")
species = pd.read_csv("burung_species.csv")
loc = pd.read_csv("location.csv")


In [3]:
for c in ["jenis_burung", "nama_ilmiah", "lokasi", "waktu", "cuaca"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

for c in ["jenis_burung", "nama_ilmiah"]:
    species[c] = species[c].astype(str).str.strip()

loc["lokasi"] = loc["lokasi"].astype(str).str.strip()

In [4]:


# pastikan tipe angka konsisten
df["titik"] = pd.to_numeric(df["titik"], errors="coerce").astype("Int64")
loc["titik"] = pd.to_numeric(loc["titik"], errors="coerce").astype("Int64")

# koordinat -> float dengan pembulatan biar aman join (kalau ada beda 0.0000001)
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce").round(6)
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce").round(6)
loc["latitude"] = pd.to_numeric(loc["latitude"], errors="coerce").round(6)
loc["longitude"] = pd.to_numeric(loc["longitude"], errors="coerce").round(6)

# ====== 3) Map species_id ======
df = df.merge(
    species[["species_id", "jenis_burung", "nama_ilmiah"]],
    on=["jenis_burung", "nama_ilmiah"],
    how="left"
)

# ====== 4) Map point_id ======
df = df.merge(
    loc[["point_id", "latitude", "longitude", "lokasi", "titik"]],
    on=["latitude", "longitude", "lokasi", "titik"],
    how="left"
)

# ====== 5) Convert tanggal & jam (optional tapi recommended) ======
# tanggal contoh: "15 July 2025" -> 2025-07-15
if "tanggal" in df.columns:
    df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce").dt.date

# jam contoh: "08:26" tetap string HH:MM (Supabase TIME bisa terima)
if "jam" in df.columns:
    df["jam"] = df["jam"].astype(str).str.strip()

# ====== 6) Build burung_bio ======
burung_bio = df[[
    "point_id",
    "species_id",
    "tanggal",
    "jam",
    "waktu",
    "cuaca",
    "jumlah_burung",
    "keterangan"
]].copy()

# observation_id kalau mau dibuat di CSV (kalau di DB auto identity, boleh nggak dipakai)
burung_bio.insert(0, "observation_id", range(1, len(burung_bio) + 1))

# ====== 7) Cek yang gagal termapping ======
missing_species = df[df["species_id"].isna()][["jenis_burung", "nama_ilmiah"]].drop_duplicates()
missing_point = df[df["point_id"].isna()][["latitude", "longitude", "lokasi", "titik"]].drop_duplicates()

print("Missing species mapping:", len(missing_species))
print(missing_species.head(10))
print("Missing location mapping:", len(missing_point))
print(missing_point.head(10))

# ====== 8) Export ======
burung_bio.to_excel("burung_bio.xlsx", index=False)
print("Saved: burung_bio.csv")


Missing species mapping: 3
               jenis_burung        nama_ilmiah
19                 Perkutut   Geopelia striata
31             Kokokan Laut  Butorides striata
468  Dara Laut Tenguk Hitam   Sterna sumatrana
Missing location mapping: 1
     latitude  longitude lokasi  titik
614   -7.3786        NaN  Dalam      4
Saved: burung_bio.csv
